<a href="https://colab.research.google.com/github/Yousef-Shihade/information-retrieval-course-tasks/blob/main/Task_03_Rocchio_Relevance_Feedback/Rocchio_Algorithm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Task 03: Rocchio Algorithm for Relevance Feedback
# Yousef Shihade

import numpy as np
import pandas as pd

rng = np.random.default_rng(seed=42)  # reproducibility

vocab = [
    "team", "coach", "hockey", "baseball", "soccer",
    "penalty", "score", "win", "loss", "season"
]
V = len(vocab)
num_docs = 50

print(" voc size : ", V)

# Randomly decide if each word appears (0 / 1)
appear_matrix = rng.integers(0, 2, size=(num_docs, V))

# If word appears the random idf 2 to 6
idf_values = rng.uniform(2.0, 6.0, size=(num_docs, V))

# final TF-IDF docs
docs = appear_matrix * idf_values

# Put docs in a DataFrame
df_docs = pd.DataFrame(docs, columns=vocab)
df_docs.index.name = "DocID"
print("\n First 5 docs (TF-IDF values) : ")
display(df_docs.head())


num_relevant = int(0.20 * num_docs)  # 10 docs
relevant_indices = rng.choice(num_docs, size=num_relevant, replace=False)

# mask: True = relevant, False = non-relevant
relevant_mask = np.zeros(num_docs, dtype=bool)
relevant_mask[relevant_indices] = True

labels = np.where(relevant_mask, "Relevant", "Non-relevant")
df_docs["Label"] = labels
print("\n Docs with labels (first 10) : ")
display(df_docs.head(10))


docs_R = docs[relevant_mask]
docs_NR = docs[~relevant_mask]

mu_R = docs_R.mean(axis=0)
mu_NR = docs_NR.mean(axis=0)

q_opt = 2 * mu_R - mu_NR   # Rocchio

df_qopt = pd.DataFrame({
    "term": vocab,
    "q_opt_value": q_opt
}).set_index("term")
print("\nq_opt vector as table:")
display(df_qopt)


sorted_indices = np.argsort(q_opt)[::-1]  # descending order
top_k = 5
top_indices = sorted_indices[:top_k]

df_top_features = df_qopt.iloc[top_indices].copy()
df_top_features["rank"] = range(1, top_k + 1)

print("\n Top 5 features in q_opt : ")
display(df_top_features[["rank", "q_opt_value"]])


def cosine_similarity(a, b):
    na = np.linalg.norm(a)
    nb = np.linalg.norm(b)
    if na == 0 or nb == 0:
        return 0.0
    return float(np.dot(a, b) / (na * nb))

# compute similarities
similarities = np.zeros(num_docs)
for i in range(num_docs):
    similarities[i] = cosine_similarity(q_opt, docs[i])

closest_docs = np.argsort(similarities)[::-1][:3]

rows = []
for rank, doc_idx in enumerate(closest_docs, start=1):
    rows.append({
        "rank": rank,
        "DocID": doc_idx,
        "Label": labels[doc_idx],
        "cosine_similarity": similarities[doc_idx]
    })
df_top_docs = pd.DataFrame(rows).set_index("rank")
print("\n Top 3 docs closest to q_opt (metadata) : ")
display(df_top_docs)

print("\n vectors of the top 3 docs:")
display(df_docs.loc[closest_docs])

 voc size :  10

 First 5 docs (TF-IDF values) : 


,team,coach,hockey,baseball,soccer,penalty,score,win,loss,season
DocID,,,,,,,,,,
0,0.000000,5.469963,3.815588,0.000000,0.000000,4.984057,0.000000,2.421112,0.00000,0.000000
1,2.584693,5.298657,3.241339,2.575488,5.683882,2.662127,3.138880,0.000000,2.46196,0.000000
2,2.221582,0.000000,0.000000,4.364575,4.722858,3.574522,0.000000,4.018105,5.50002,0.000000
3,0.000000,0.000000,0.000000,2.997550,4.284931,0.000000,2.197016,3.494457,0.00000,2.406688
4,0.000000,2.207847,5.699367,0.000000,0.000000,5.610613,0.000000,5.208104,5.11791,4.569933



 Docs with labels (first 10) : 


,team,coach,hockey,baseball,soccer,penalty,score,win,loss,season,Label
DocID,,,,,,,,,,,
0,0.000000,5.469963,3.815588,0.000000,0.000000,4.984057,0.000000,2.421112,0.000000,0.000000,Non-relevant
1,2.584693,5.298657,3.241339,2.575488,5.683882,2.662127,3.138880,0.000000,2.461960,0.000000,Non-relevant
2,2.221582,0.000000,0.000000,4.364575,4.722858,3.574522,0.000000,4.018105,5.500020,0.000000,Non-relevant
3,0.000000,0.000000,0.000000,2.997550,4.284931,0.000000,2.197016,3.494457,0.000000,2.406688,Non-relevant
4,0.000000,2.207847,5.699367,0.000000,0.000000,5.610613,0.000000,5.208104,5.117910,4.569933,Non-relevant
5,5.115985,0.000000,0.000000,0.000000,0.000000,0.000000,3.540358,0.000000,3.065853,2.559074,Non-relevant
6,3.911509,3.667557,0.000000,3.470047,0.000000,0.000000,3.517856,0.000000,0.000000,0.000000,Non-relevant
7,5.665392,0.000000,0.000000,0.000000,5.394242,0.000000,0.000000,0.000000,4.531671,3.152622,Non-relevant
8,4.939573,0.000000,0.000000,5.442876,2.528411,4.457519,0.000000,0.000000,2.337973,5.743759,Non-relevant



q_opt vector as table:


...

,rank,q_opt_value
term,,
hockey,1,3.923727
season,2,3.250742
soccer,3,2.915928
score,4,2.254192
win,5,2.119792



 Top 3 docs closest to q_opt (metadata) : 


,DocID,Label,cosine_similarity
rank,,,
1,42,Relevant,0.862116
2,10,Relevant,0.854707
3,12,Non-relevant,0.830876



 vectors of the top 3 docs:


,team,coach,hockey,baseball,soccer,penalty,score,win,loss,season,Label
DocID,,,,,,,,,,,
42,2.547285,4.299488,5.990320,4.803524,0.000000,0.000000,5.661195,3.987666,0.000000,3.461514,Relevant
10,4.464663,0.000000,4.259802,0.000000,3.863941,4.090527,5.055694,5.196979,0.000000,4.398374,Relevant
12,0.000000,3.082140,3.456768,0.000000,2.630447,2.591133,0.000000,3.751616,3.533279,4.918743,Non-relevant
